# Criterium Research API Tutorial
This notebook will walk you through interacting with the FastAPI application we just built. 

**Prerequisites:**
1. Ensure your server is running (`make run` in a terminal).
2. Ensure your `.env` file contains valid API keys for Gemini and Firecrawl.

In [ ]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"

def print_json(data):
    print(json.dumps(data, indent=2))

## 1. Create a Research Collection
Since we are using an in-memory SQLite database (`:memory:`), each server restart begins with an empty database. Let's create our test collection first using the Book Series schema.
Notice that we pass the raw JSON schema directly into the `llm_schema` field.

In [ ]:
schema = {
  "type": "object",
  "properties": {
    "title": {
      "type": "string",
      "description": "The exact title of the primary book."
    },
    "year": {
      "type": "integer",
      "description": "The 4-digit year the book was published."
    },
    "series_books": {
      "type": "array",
      "description": "Other books in the same series.",
      "items": {
        "type": "object",
        "properties": {
          "title": { "type": "string" },
          "chronological_order": { "type": "integer" }
        },
        "required": ["title"]
      }
    }
  },
  "required": ["title", "year"]
}

payload = {
    "name": "Book Series Researcher",
    "extraction_prompt": "You are a literary researcher. Extract the details of the primary book discussed on this page, as well as a list of any other books mentioned that belong to the same series.",
    "llm_schema": schema
}


response = requests.post(f"{BASE_URL}/collections", json=payload)
print("Status Code:", response.status_code)
print("Response text:", response.text)
response.raise_for_status()
collection_data = response.json()
collection_id = collection_data.get("id")
print(f"Created collection with ID: {collection_id}")
print_json(collection_data)


## 2. Verify / List All Collections
Now let's query `GET /collections` to verify our newly created collection is listed in the database.

In [ ]:
response = requests.get(f"{BASE_URL}/collections")
print("Status Code:", response.status_code)
print_json(response.json())

## 3. Extract a Product (The Research Protocol)
Now we'll trigger the 4-stage research pipeline. We'll pass a search query to our endpoint.
Behind the scenes, this will:
1. Search and Scrape using Firecrawl
2. Save the raw markdown to SQLite
3. Run the markdown against Gemini using our JSON Schema
4. Save the structured JSON back to SQLite

In [ ]:
# Note: This will fail if your .env keys are still set to "placeholder"
query_payload = {
    "product_info": "The Lord of the Rings: The Fellowship of the Ring book"
}

# This request might take a few seconds as it hits both APIs
print(f"Triggering extraction for collection {collection_id}...")
extract_response = requests.post(f"{BASE_URL}/collections/{collection_id}/products", json=query_payload)

print("Status Code:", extract_response.status_code)
print_json(extract_response.json())

## 4. View the Extracted Data
Finally, let's pull down the products we just researched to see the raw markdown and the structured JSON output!

In [ ]:
products_response = requests.get(f"{BASE_URL}/collections/{collection_id}/products")

for product in products_response.json():
    print("="*50)
    print("PRODUCT:", product["name"])
    print("URL:", product["source_url"])
    print("-" * 50)
    print("EXTRACTED DATA:")
    print_json(product["extracted_data"])
    print("="*50)